In [1]:

import numpy as np
import pandas as pd
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import time
import cv2
import mediapipe as mp
from ultralytics import YOLO
from ultralytics.utils.plotting import Annotator
from tensorflow.keras.preprocessing.image import  img_to_array

In [2]:
LEFT_EYE = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33, 160, 158, 133, 153, 144]

EAR_THRESHOLD = 0.20
CLOSED_TIME_THRESHOLD = 4

eye_close_time = {}

def eye_openness(landmarks, eye_points, w, h):
    p = [(int(landmarks[i].x * w), int(landmarks[i].y * h)) for i in eye_points]

    v1 = np.linalg.norm(np.array(p[1]) - np.array(p[5]))
    v2 = np.linalg.norm(np.array(p[2]) - np.array(p[4]))
    h_dist = np.linalg.norm(np.array(p[0]) - np.array(p[3]))

    return (v1 + v2) / (2.0 * h_dist)

In [3]:
def check_focusing(ROI, track_id):
    global face_mesh, x11, y11, r, c

    ROI.flags.writeable = False
    result = face_mesh.process(ROI)
    ROI.flags.writeable = True

    rows, cols, _ = ROI.shape

    face_3d = []
    face_2d = []

    # default
    eye_closed = False

    if result.multi_face_landmarks:

        for facelms in result.multi_face_landmarks:

            # ================= EYE CHECK =================
            left_eye = eye_openness(facelms.landmark, LEFT_EYE, cols, rows)
            right_eye = eye_openness(facelms.landmark, RIGHT_EYE, cols, rows)

            eye_ratio = (left_eye + right_eye) / 2

            if eye_ratio < EAR_THRESHOLD:
                if track_id not in eye_close_time:
                    eye_close_time[track_id] = time.time()
                if time.time() - eye_close_time[track_id] >= CLOSED_TIME_THRESHOLD:
                    eye_closed = True
                else:
                    eye_closed = False
            else:
                eye_close_time[track_id] = time.time()
                eye_closed = False

            # ================= HEAD POSE (UNCHANGED LOGIC) =================
            for id, lm in enumerate(facelms.landmark):
                a, b = int(lm.x * cols), int(lm.y * rows)

                x, y = int(a + x11), int(b + y11)
                face_2d.append([x, y])
                face_3d.append([x, y, lm.z])

            face_2d = np.array(face_2d, dtype=np.float64)
            face_3d = np.array(face_3d, dtype=np.float64)

            focal_len = 1 * c
            cam_matrix = np.array([
                [focal_len, 0, r / 2],
                [0, focal_len, c / 2],
                [0, 0, 1]
            ])

            dist_matrix = np.zeros((4, 1), dtype=np.float64)

            success, rot_vec, trans_vec = cv2.solvePnP(
                face_3d, face_2d, cam_matrix, dist_matrix
            )

            rmat, _ = cv2.Rodrigues(rot_vec)
            angles, _, _, _, _, _ = cv2.RQDecomp3x3(rmat)

            x_angle = angles[0] * 360
            y_angle = angles[1] * 360

            # ================= HEAD NOT FOCUS =================
            head_not_focused = (
                y_angle < -5 or y_angle > 10 or
                x_angle < -10 or x_angle > 20
            )

            # ================= FINAL DECISION =================
            if eye_closed or head_not_focused:
                return "Not Focusing"
            else:
                return "Normal"

    return "Normal"

In [4]:
def check_cheating(ROI):
    ROI.flags.writeable = False
    posingEstimation = face_mesh.process(ROI)
    ROI.flags.writeable = True
    
    rows, cols, _ =ROI.shape
    face_3d = []
    face_2d = []
    #print(results.multi_face_landmarks)
    if posingEstimation.multi_face_landmarks:
        for facelms in posingEstimation.multi_face_landmarks:
            
            for id ,lm in enumerate(facelms.landmark):
                a,b=int(lm.x*cols),int(lm.y*rows)
                if id in [1] :
                    #cv2.circle(frame,(a+x11,b+y11),10,(0,0,255),-1)
                    
                    nose_2d=(a+x11 , b+y11)
                    nose_3d=(a+x11 , b+y11 , lm.z * 5000)
                x , y = int(a+x11), int(b+y11) 
                face_2d.append([x,y])
                face_3d.append([x,y,lm.z])
            face_2d = np.array(face_2d, dtype=np.float64)
            face_3d = np.array(face_3d, dtype=np.float64)
            focal_len = 1* c
            cam_matrix = np.array([ [focal_len , 0 , r/2],
                                    [0 , focal_len , c/2],
                                    [0,0,1]])
            dist_matrix= np.zeros((4,1), dtype=np.float64)
            success , rot_vec , trans_vec =cv2.solvePnP(face_3d, face_2d , cam_matrix , dist_matrix) 
            rmat , jac= cv2.Rodrigues(rot_vec)
            angles , mtxR , mtxQ , Qx , Qy , Qz= cv2.RQDecomp3x3(rmat)
            x=angles[0] * 360
            y=angles[1] * 360 
            z=angles[2] * 360
            # y يمثل اتجاه الرأس نحو اليمين أو اليسار
            # x يمثل اتجاه الرأس لأعلى أو لأسفل
            # z يمثل دوران الرأس حول المحور الأمامي (مثل حركة الالتفاف).

            if y < -5: # قيمة سالبة تعني أن الرأس يميل لليسار.
                text = "Cheating"
            elif y > 10: # قيمة موجبة تعني أن الرأس يميل لليمين.
                 text = "Cheating"
            elif x < -20: # قيمة سالبة تعني أن الرأس يميل لأسفل.
                text = "Normal"
            elif x > 20: # قيمة موجبة تعني أن الرأس يميل لأعلى.
                text = "Normal"  
            else:
                 text = "Normal" 
            return text
    return "Normal" 

In [18]:

def process_track(text, track_id):
    if text in ["Not Focusing", "Cheating"]:
        if track_id not in time_temp:
            # If the track_id doesn't exist, add it with the current time in milliseconds
            time_temp[track_id] = int(time.time()) 
   
    elif text == "Normal":
        # Calculate the difference between the current time and the previous cheating time
        if track_id in time_temp:
            current_time = int(time.time())
            time_difference = current_time - time_temp[track_id]
            # Update the time for the track_id
            if track_id in time_id:
                time_id[track_id] += time_difference 
            else:
                time_id[track_id] = time_difference 
            time_temp.pop(track_id)
    else:
        print("Invalid text input. Please use 'Not Focusing' or 'Normal'.")


In [ ]:

df=pd.read_csv('Face Recognition.csv')

In [7]:
df.head(1)

,0,1,2,3,4,5,6,7,8,9,...,119,120,121,122,123,124,125,126,127,128
0,-0.040733,0.100574,0.073227,0.032741,0.058872,0.086022,0.062363,-0.031478,-0.09761,0.126788,...,0.183315,0.044654,-0.047918,-0.06798,0.112717,-0.028919,0.087343,0.085478,-0.037088,Mahmoud


In [8]:
# libraries
from Openface import load_model

###############################################
# load OpenFace Model  
model_openface=load_model()
###############################################
# Preprocessing of image to predict the name of student
def preprocess_image(img):
    img = img_to_array(img)
    img = img/255.0
    img = np.expand_dims(img, axis=0)
    return img
###############################################
# to identify the student
def Face_Recognition(roi):
    roi = cv2.cvtColor(roi,cv2.COLOR_BGR2RGB)
    roi=cv2.resize(roi,dsize=(96,96),interpolation=cv2.INTER_CUBIC)
    roi=preprocess_image(roi)
    embedding_vector = model_openface.predict(roi)[0]
    return embedding_vector

In [9]:
def findCosineSimilarity(source_representation, test_representation):
    a = np.matmul(np.transpose(source_representation), test_representation)
    b = np.sum(np.multiply(source_representation, source_representation))
    c = np.sum(np.multiply(test_representation, test_representation))
    return 1 - (a / (np.sqrt(b) * np.sqrt(c)))

In [16]:
mode = "Cheating" # "Focusing" or "Cheating"

In [ ]:
mpFacemesh = mp.solutions.face_mesh
face_mesh = mpFacemesh.FaceMesh(
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5)


# Initialize YOLO model
model = YOLO('YoloV8 Face.pt')
names = model.model.names

# Load video
cap = cv2.VideoCapture(0)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Set video codec and frame rate
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('outputz.mp4', fourcc, 30, (width, height))  # Adjust frame rate (30 fps in this example)

# Variables
time_id = {}
time_temp = {}
student_id={}
            
while cap.isOpened():
    success, frame = cap.read()

    if not success:
        print("Video frame is empty or video processing has been successfully completed.")
        break
    r, c, _ = frame.shape
    annotator = Annotator(frame, line_width=2)

    # Run YOLO inference
    results = model.track(frame, iou=0.3, show=False, tracker="bytetrack.yaml", persist=True, verbose=False)

    if results[0].boxes.id is not None:
        track_ids = results[0].boxes.id.int().cpu().tolist()
        boxes = results[0].boxes.xyxy.cpu()
        conf = results[0].boxes.conf.tolist()

        for box, track_id, cof in zip(boxes, track_ids, conf):
            x1, y1, x2, y2 = box.int().tolist()
            face = frame[y1:y2, x1:x2]
            if track_id not in student_id :
                emded=Face_Recognition(face)
                for i in range(df.shape[0]):
                    sim=findCosineSimilarity(emded,df.iloc[i,:-1])
                    print(sim)
                    if sim < 0.2 :
                        name = df.iloc[i,-1]
                        cv2.putText(frame,str(name) ,(x1,y1-8), cv2.FONT_HERSHEY_SIMPLEX,1,(255, 0, 0), 2, cv2.LINE_AA)
                        student_id[track_id] = name
                        break
                else :
                        student_id[track_id] = 'Unknown'
                        cv2.putText(frame,'Unknown' ,(x1,y1-8), cv2.FONT_HERSHEY_SIMPLEX,1,
                                    (255, 0, 0), 2, cv2.LINE_AA)
            annotator.box_label(box, label=str(track_id), color=(0, 0, 255)) 
                
            if track_id in student_id : 
                y11 = y1 - 150
                x11 = x1 - 150
                y22 = y2 + 150
                x22 = x2 + 150
                ROI = frame[y11:y22, x11:x22]
                if mode == "Focusing":
                    if ROI.size != 0 :
                        ROI = cv2.cvtColor(ROI,cv2.COLOR_BGR2RGB)
                        text = check_focusing(ROI, track_id)
                        
                        if track_id not in time_id:
                            time_id[track_id] = 0
                        process_track(text, track_id)
                        
                        string = str (student_id[track_id]) +' '+str(time_id[track_id])+' sec'
                        if text == 'Not Focusing':
                            annotator.box_label(box, label=str(string), color=(0, 0, 255))
                        else :
                            annotator.box_label(box, label=str(string), color=(255, 0, 0))
                    for ID, TIME in time_id.items():
                        cv2.putText(frame, f'{student_id[ID]} Not Focusing {TIME} Second', (10, 100 + ID * 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                else:
                    if ROI.size != 0 :
                        ROI = cv2.cvtColor(ROI,cv2.COLOR_BGR2RGB)
                        text = check_cheating(ROI)
                    
                        if track_id not in time_id:
                            time_id[track_id] = 0
                        process_track(text, track_id)
                    
                        string = str (student_id[track_id]) +' '+str(time_id[track_id])+' sec'
                        if text == 'Cheating':
                            annotator.box_label(box, label=str(string), color=(0, 0, 255))
                        else :
                            annotator.box_label(box, label=str(string), color=(255, 0, 0))
                    for ID, TIME in time_id.items():
                        cv2.putText(frame, f'{student_id[ID]} Cheating {TIME} Second', (10, 100 + ID * 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
            
    # Write frame to output video
    out.write(frame)

    # Display frame
    cv2.imshow('img', frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

# Release video capture and writer
cap.release()
out.release()
cv2.destroyAllWindows()


I0000 00:00:1777924870.127612  245850 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4
W0000 00:00:1777924870.129186  264965 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777924870.133124  264965 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
OpenCV: FFMPEG: tag 0x44495658/'XVID' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'


1/1 [==============================] - 0s 10ms/step
0.6105263211584004
0.17144793145367887


In [20]:
student_id

{1: 'Mahmoud'}

In [ ]:
time_id

{1: 5}

: 